# Módulo 2 - Bloco da tarde: notebook de apoio

Programa Data Science for Decision Makers - Ambev (DSDMT10). Caso Coroa Premium.

Uso em sala: este notebook é o fallback dos hands-on das 14h45 e das 16h15. Se a ferramenta de IA de alguma mesa travar, o professor projeta e executa estas células. Todas as células rodam offline, sobre `dados/Coroa_Premium_PDV_Performance.csv`.

Sequência: carga do dado, quadro descritivo, teste da H1 (cross-tab por concorrentes), célula com erro proposital para o exercício de debugging, teste da H2 (visitas x delta por tier), gráfico e resumo executivo.

In [ ]:
from pathlib import Path

import pandas as pd

CSV = Path("../dados/Coroa_Premium_PDV_Performance.csv")
if not CSV.exists():
    CSV = Path("Coroa_Premium_PDV_Performance.csv")  # caso o notebook rode solto

df = pd.read_csv(CSV)
print(f"{len(df)} PDVs, {len(df.columns)} colunas")
df.head(3)

## 1. Quadro descritivo (o que a Beatriz montou em três horas)

Atenção ao primeiro resultado: "delta médio" tem duas definições legítimas e os números são diferentes. A média simples trata todos os PDVs com o mesmo peso; a variação do volume total pondera pelo tamanho de cada PDV.

In [ ]:
media_simples = df["delta_volume_yoy_pct"].mean()
agregado = (df["volume_q4_2025_hl"].sum() / df["volume_q4_2024_hl"].sum() - 1) * 100
abaixo_15 = (df["delta_volume_yoy_pct"] < -15).mean() * 100
tier_a = df[df["tier_historico"] == "A"]
share_a_pdvs = len(tier_a) / len(df) * 100
share_a_vol = tier_a["volume_q4_2024_hl"].sum() / df["volume_q4_2024_hl"].sum() * 100

print(f"Delta médio (média simples por PDV): {media_simples:+.1f}%")
print(f"Delta do volume total (ponderado):   {agregado:+.1f}%")
print(f"PDVs com delta abaixo de -15%:       {abaixo_15:.1f}%")
print(f"Tier A: {share_a_pdvs:.0f}% dos PDVs e {share_a_vol:.0f}% do volume")

df.groupby("regiao").apply(
    lambda g: (g["volume_q4_2025_hl"].sum() / g["volume_q4_2024_hl"].sum() - 1) * 100,
    include_groups=False,
).round(1).rename("delta_agregado_pct").to_frame()

## 2. H1: a queda é uniforme ou concentrada?

Cruzamento de `delta_volume_yoy_pct` por faixa de `concorrentes_premium_no_pdv` (hipótese 1 do memorando de Daniel Yamada, Exhibit 3).

In [ ]:
faixa = pd.cut(
    df["concorrentes_premium_no_pdv"], bins=[-1, 1, 2, 5], labels=["0-1", "2", "3+"]
)
h1 = df.groupby(faixa, observed=True).agg(
    n_pdvs=("pdv_id", "count"),
    delta_medio_pct=("delta_volume_yoy_pct", "mean"),
    pct_abaixo_15=("delta_volume_yoy_pct", lambda s: (s < -15).mean() * 100),
).round(1)
h1

Leitura esperada: a queda está concentrada nos PDVs com 3 ou mais concorrentes ativados. PDVs com 0 ou 1 concorrente caem pouco. Implicação: a tese de "canal afundando de forma uniforme" (Caminho B) não descreve o dado; a concentração torna a segmentação do Caminho A verificável.

## 3. Exercício de debugging: célula com erro proposital

A célula abaixo falha com `KeyError`. Procedimento do exercício: executar, copiar o traceback completo, colar na IA com o pedido "explique este erro para quem nunca programou e corrija o script". A causa: a coluna se chama `visitas_promotor_mes`, não `visitas_promotor`.

In [ ]:
# ERRO PROPOSITAL: nao corrigir antes de rodar; o traceback e o material do exercicio
df_c = df[df["tier_historico"] == "C"]
correl_c = df_c["visitas_promotor"].corr(df_c["delta_volume_yoy_pct"])
print(correl_c)

## 4. H2: a visita do promotor converte em volume?

Versão corrigida: correlação entre `visitas_promotor_mes` e `delta_volume_yoy_pct`, por tier (hipótese 2 do Exhibit 3).

In [ ]:
h2 = (
    df.groupby("tier_historico")
    .apply(
        lambda g: g["visitas_promotor_mes"].corr(g["delta_volume_yoy_pct"]),
        include_groups=False,
    )
    .round(2)
    .rename("correlacao_visitas_x_delta")
    .to_frame()
)
h2

Leitura esperada: correlação positiva no tier A (visita acompanha resposta), próxima de zero a moderada no B, e negativa no tier C: mais visita não significa mais volume nesse grupo. Implicação operacional: parte da rota de promotores em tier C não gera resposta, o que é compatível com a estimativa operacional de Rodrigo (PDVs que não deveriam ter entrado no programa), sem confirmar o número exato de 30%.

In [ ]:
import matplotlib.pyplot as plt

fig, eixos = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for eixo, tier in zip(eixos, ["A", "B", "C"]):
    sub = df[df["tier_historico"] == tier]
    eixo.scatter(sub["visitas_promotor_mes"], sub["delta_volume_yoy_pct"], s=6, alpha=0.3)
    eixo.axhline(0, linewidth=0.8)
    eixo.set_title(f"Tier {tier}")
    eixo.set_xlabel("Visitas do promotor por mês")
eixos[0].set_ylabel("Delta de volume YoY (%)")
fig.suptitle("H2: visitas x delta por tier")
plt.tight_layout()
plt.show()

## 5. Do resultado ao texto executivo

A célula abaixo monta o rascunho do e-mail com os números calculados. Em sala, o passo seguinte é colar esse rascunho na IA com o prompt do slide "Do script ao relatório executivo" (máximo 12 linhas, três evidências com origem, uma ressalva).

In [ ]:
h1_01 = h1.loc["0-1", "delta_medio_pct"]
h1_3p = h1.loc["3+", "delta_medio_pct"]
corr_a = h2.loc["A", "correlacao_visitas_x_delta"]
corr_c = h2.loc["C", "correlacao_visitas_x_delta"]

rascunho = f"""Achados para a recomendação (origem: Coroa_Premium_PDV_Performance.csv):
1. A queda não é uniforme: PDVs com 3+ concorrentes caem {h1_3p:.1f}% em média;
   PDVs com 0-1 concorrente caem {h1_01:.1f}% (colunas delta_volume_yoy_pct
   e concorrentes_premium_no_pdv).
2. Volume total do programa: {agregado:+.1f}% YoY; média simples por PDV:
   {media_simples:+.1f}% (a diferença vem do peso dos PDVs grandes).
3. Visita de promotor: correlação com delta de {corr_a:+.2f} no tier A e
   {corr_c:+.2f} no tier C (colunas visitas_promotor_mes e delta_volume_yoy_pct).
Ressalva: o dado não permite atribuir causa; concorrência e queda podem ter
causa comum não observada."""
print(rascunho)